# 02 - Preprocessing: criterios desde el diccionario de datos

Este notebook documenta criterios de preprocesamiento derivados del diccionario oficial del dataset de siniestros viales. No modifica el dataset original y no reemplaza los pipelines existentes; su funcion es explicitar decisiones de interpretacion antes de aplicar transformaciones productivas.

## Data Understanding / Diccionario de Datos

El preprocesamiento debe partir de la metadata oficial. Las variables no son solo columnas tecnicas: representan conceptos producidos por una institucion, con reglas de registro, categorias definidas y codigos de ausencia de informacion. Incorporar este conocimiento de dominio mejora la calidad analitica porque reduce errores de codificacion, evita comparaciones inadecuadas y vuelve reproducibles las decisiones sobre faltantes.

In [3]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import (
    cargar_dataset,
    GRAVEDAD_VICTIMA_ORDER,
    OFFICIAL_CATEGORY_DEFINITIONS,
    OFFICIAL_DATA_DICTIONARY,
    SD_MEANING,
    SD_VALUE,
)

from src.preprocessing import clean_data

pd.DataFrame(OFFICIAL_DATA_DICTIONARY.items(), columns=["variable", "significado_oficial"])

,variable,significado_oficial
0,id_siniestro,Identificador unico del siniestro.
1,fecha_siniestro,Fecha en formato aaaa-mm-dd en la que sucedio ...
2,anio_siniestro,Anio del siniestro.
3,modo_desplazamiento_victima,Vehiculo que ocupaba quien haya fallecido o se...
4,sexo_victima,Sexo de la victima informado por fuente policial.
5,edad_victima,Edad de la victima al momento del siniestro.
6,gravedad_victima,Nivel maximo conocido de gravedad de la lesion...
7,rol_victima,Posicion relativa al vehiculo que presentaba l...
8,fecha_fallecimiento_victima,Fecha de fallecimiento de las victimas mortales.


## Criterios de tratamiento

- `SD` significa **Sin Datos** y debe tratarse como metadata de ausencia, no como categoria real del fenomeno vial.
- `gravedad_victima` debe conservar el orden `LEVE < GRAVE < MORTAL` cuando se construyan variables derivadas o visualizaciones.
- Las categorias de `modo_desplazamiento_victima` y `rol_victima` deben interpretarse segun sus definiciones institucionales.
- Cualquier imputacion, exclusion o recodificacion debe registrarse como decision analitica, indicando su impacto potencial sobre resultados y sesgos.

In [4]:
pd.DataFrame({
    "categoria": list(GRAVEDAD_VICTIMA_ORDER.keys()),
    "orden_ordinal": list(GRAVEDAD_VICTIMA_ORDER.values()),
    "definicion_institucional": [
        OFFICIAL_CATEGORY_DEFINITIONS["gravedad_victima"][categoria]
        for categoria in GRAVEDAD_VICTIMA_ORDER
    ],
})

,categoria,orden_ordinal,definicion_institucional
0,LEVE,1,Personas lesionadas que reciben el alta medica...
1,GRAVE,2,Toda persona cuya lesion exige la hospitalizac...
2,MORTAL,3,Victima que fallece dentro de los 30 dias de p...


## Riesgos de interpretacion incorrecta

Un preprocesamiento automatico que convierta categorias a numeros sin revisar el diccionario puede introducir errores conceptuales. Por ejemplo, ordenar alfabeticamente `GRAVE`, `LEVE` y `MORTAL` no respeta la severidad real. A su vez, convertir `SD` en una clase comun puede hacer que la ausencia de informacion parezca un atributo de la victima.

La metadata funciona como puente entre calidad de datos y calidad analitica: permite distinguir faltantes, categorias validas, categorias residuales y niveles ordinales. En consecuencia, el conocimiento de dominio no es un agregado narrativo, sino una condicion para que las transformaciones sean interpretables.

## Carga del dataset fuente

Se carga el archivo disponible en `data/raw/` unicamente en modo lectura. Esta etapa no escribe, mueve ni sobrescribe archivos originales. La salida procesada se persistira exclusivamente en `data/processed/`.

In [ ]:
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

raw_candidates = sorted(
    [
        path
        for path in RAW_DATA_DIR.iterdir()
        if path.suffix.lower() in {".csv", ".xlsx", ".xls"}
    ]
)

if not raw_candidates:
    raise FileNotFoundError(
        "No se encontro un dataset fuente en data/raw/. "
        "Coloque el archivo original alli antes de ejecutar el preprocessing."
    )

RAW_DATA_FILE = raw_candidates[0]
df = cargar_dataset(RAW_DATA_FILE)

print(f"Dataset fuente cargado desde: {RAW_DATA_FILE.resolve()}")
print(f"Shape inicial: {df.shape}")

## Preprocesamiento conservador

Se genera `df_limpio` como copia procesada para analisis posteriores. Las transformaciones son deliberadamente acotadas: se conserva la semantica oficial de `SD`, se normaliza el nombre de `gravedad_victima` cuando el archivo lo trae con una variante de mayusculas, y se recortan espacios accidentales en variables textuales. Estas operaciones no modifican `data/raw/`.

In [ ]:
df_limpio = clean_data(df)

if "gravedad_victima" not in df_limpio.columns and "GRAVEdad_victima" in df_limpio.columns:
    df_limpio = df_limpio.rename(columns={"GRAVEdad_victima": "gravedad_victima"})

columnas_texto = df_limpio.select_dtypes(include=["object", "string"]).columns
for columna in columnas_texto:
    df_limpio[columna] = df_limpio[columna].astype("string").str.strip()

if "gravedad_victima" in df_limpio.columns:
    df_limpio["gravedad_victima"] = pd.Categorical(
        df_limpio["gravedad_victima"].astype("string").str.upper(),
        categories=list(GRAVEDAD_VICTIMA_ORDER.keys()),
        ordered=True,
    )

print(f"Shape luego del preprocesamiento conservador: {df_limpio.shape}")
display(df_limpio.head())

## Guardado del dataset procesado

La carpeta `data/processed/` se crea con `pathlib` si no existe. El archivo final se guarda como `siniestros_limpio.csv`; no se realiza ninguna escritura sobre `data/raw/`.

In [ ]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = PROCESSED_DATA_DIR / "siniestros_limpio.csv"
df_limpio.to_csv(OUTPUT_FILE, index=False)

print(f"Dataset procesado guardado en: {OUTPUT_FILE.resolve()}")
print(f"Shape final: {df_limpio.shape}")